# S09 · Automated Rule Library Construction via SynKit (Local USPTO-50K)

**Pipeline:** `USPTO_50K.csv → RXNMapper atom maps → canonicalized AAM → ITS + reaction center → WL-prefilter + exact clustering → exported rule library`

<div style="padding: 10px; border-left: 4px solid #4c72b0; background:#f7f7f7">
<b>Goal.</b> Build a <b>deterministic</b>, <b>deduplicated</b> reaction-rule library from a real benchmark dataset, using SynKit wherever possible.<br>
If SynKit APIs are not available in your environment, the notebook falls back to lightweight reference implementations compatible with S01–S08.
</div>

**Input file (provided in this repo):**

- `synedu/S09/data/USPTO_50K.csv` with columns: `id, class, reactions` where `reactions` are **unmapped** `reactants>>products` SMILES.

**Outputs (written by this notebook):**

- `synedu/S09/out/uspto50k_mapped_*.json.gz` (cached mapped reactions)
- `synedu/S09/out/rule_library.json.gz` (clustered rule cores + metadata)
- `synedu/S09/out/rules_gml/` (optional graph exports per cluster)


## Roadmap

1. Load local USPTO-50K reactions from `synedu/S09/data/USPTO_50K.csv`
2. Atom-map reactions with **RXNMapper** (batched, cached)
3. Canonicalize mapped reactions (stable component order + stable map renumbering)
4. Convert mapped reactions to **ITS graphs**, extract **reaction centers**
5. Cluster reaction centers by **WL hash prefilter** + **exact isomorphism**
6. Export a compact **rule library** (core graphs + provenance)

> Chemist's intuition: we turn each reaction into a “what changed” graph (reaction center), then deduplicate similar centers into reusable rules.


In [ ]:
from __future__ import annotations

import gzip
import json
import math
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd
from tqdm.auto import tqdm

import networkx as nx
from networkx.algorithms import isomorphism as iso

from rdkit import Chem
import importlib.metadata as im

def _ver(pkg: str) -> str:
    try:
        return im.version(pkg)
    except Exception:
        return "not-installed"

print("rdkit:", getattr(Chem, "__version__", "unknown"))
print("networkx:", nx.__version__)
print("synkit:", _ver("synkit"))
print("rxnmapper:", _ver("rxnmapper"))

# --- paths (repo-root relative) ---
CSV_PATH = Path("./data/USPTO_50K.csv")
OUTDIR = Path("./out")
OUTDIR.mkdir(parents=True, exist_ok=True)

RULES_GML_DIR = OUTDIR / "rules_gml"
RULES_GML_DIR.mkdir(parents=True, exist_ok=True)

assert CSV_PATH.exists(), f"Missing input file: {CSV_PATH.resolve()}"
print("Using CSV:", CSV_PATH.resolve())
print("Output dir:", OUTDIR.resolve())


## 1. Load local USPTO-50K reactions

The provided CSV contains:

- `id`: document/patent id
- `class`: reaction class label (1..10 in many USPTO-50K variants)
- `reactions`: reaction SMILES, typically `reactants>>products` (unmapped)

We normalize to a table with columns:

- `rxn` : unmapped reaction SMILES
- `id`, `class`


In [ ]:
df = pd.read_csv(CSV_PATH)
# Normalize column names (robust to minor variations)
cols = {c.lower(): c for c in df.columns}
rxn_col = None
for cand in ["reactions", "reaction", "rxn", "rxn_smiles", "reaction_smiles", "smiles"]:
    if cand in cols:
        rxn_col = cols[cand]
        break
if rxn_col is None:
    raise KeyError(f"Could not find reaction SMILES column. Columns: {list(df.columns)}")

df = df.rename(columns={rxn_col: "rxn"})
if "id" not in df.columns and "ID" in df.columns:
    df = df.rename(columns={"ID": "id"})
if "class" not in df.columns and "Class" in df.columns:
    df = df.rename(columns={"Class": "class"})

df["rxn"] = df["rxn"].astype(str).str.strip()

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()


### 1.1 Basic dataset sanity check

We ensure:
- reactions contain an arrow (`>>` or `>...>`),
- RDKit can parse at least the product side for a small sample.

(We do not attempt to “fix” dataset issues here; rule libraries inherit dataset quality.)


In [ ]:
from rdkit import Chem
def split_rxn(rxn: str) -> Tuple[str, str, str]:
    """Return (reactants, reagents, products) from reaction SMILES."""
    rxn = rxn.strip()
    if rxn.count(">") >= 2:
        a, b, c = rxn.split(">", 2)
        return a, b, c
    if ">>" in rxn:
        a, c = rxn.split(">>", 1)
        return a, "", c
    return rxn, "", ""

bad = 0
ncheck = min(2000, len(df))
for x in df["rxn"].head(ncheck):
    r, g, p = split_rxn(x)
    if not p:
        bad += 1
        continue
    m = Chem.MolFromSmiles(p.split(".")[0])
    if m is None:
        bad += 1

print("Checked:", ncheck)
print("Bad in sample:", bad)


## 2. Atom mapping with RXNMapper (batched + cached)

Let \( \mathrm{rxn} = R > G > P \) be an **unmapped** reaction SMILES.

RXNMapper produces a mapped reaction \( \mathrm{rxn}^{\#} \) where atoms carry map IDs `:k`.

We cache mapped reactions to avoid recomputation.

> Tip: Start with `N=2000` to validate the pipeline, then scale `N=len(df)` for full USPTO-50K.


In [ ]:
import json
import gzip
# How many reactions to map in this run?
N = 2000  # set to len(df) for full run
BATCH = 64

cache_path = OUTDIR / f"uspto50k_mapped_N{N}.json.gz"

def _save_json_gz(obj: Any, p: Path) -> None:
    with gzip.open(p, "wt", encoding="utf-8") as f:
        json.dump(obj, f)

def _load_json_gz(p: Path) -> Any:
    with gzip.open(p, "rt", encoding="utf-8") as f:
        return json.load(f)

if cache_path.exists() and cache_path.stat().st_size > 0:
    print("Loading mapped cache:", cache_path)
    mapped = _load_json_gz(cache_path)
else:
    try:
        from rxnmapper import RXNMapper
    except Exception as e:
        raise ImportError(
            "rxnmapper is required for atom mapping. Install with `pip install rxnmapper`.\n"
            f"Import error: {e}"
        )

    rxnmapper = RXNMapper()
    rxns = df["rxn"].head(N).tolist()
    mapped = []

    for i in tqdm(range(0, len(rxns), BATCH), desc="RXNMapper"):
        batch = rxns[i : i + BATCH]
        out = rxnmapper.get_attention_guided_atom_maps(batch)
        for rec in out:
            mapped.append(rec.get("mapped_rxn", ""))

    _save_json_gz(mapped, cache_path)
    print("Saved mapped cache:", cache_path)

dfN = df.head(N).copy()
dfN["rxn_mapped_raw"] = mapped
dfN.head()


## 3. Canonicalize mapped reactions (stable AAM)

Atom-mapped reactions are **not unique**: component order and even map numbering can vary.
To build a *stable* rule library, we canonicalize mapped reactions:

\[
\mathrm{rxn}^{\#} \;\mapsto\; \mathrm{canon}(\mathrm{rxn}^{\#})
\]

We *prefer* SynKit canonicalization if available; otherwise we use a deterministic fallback that:

1. canonicalizes each molecule SMILES (keeping atom maps)
2. sorts components by canonical SMILES
3. renumbers atom-map IDs by first-appearance order (1..n)


In [ ]:
# --- Try SynKit first (preferred) ---
try:
    from synkit.Chem.Reaction.canon_rsmi import CanonRSMI  # type: ignore
    _HAS_SYNKIT_CANON = True
    print("SynKit CanonRSMI available.")
except Exception:
    _HAS_SYNKIT_CANON = False
    print("SynKit CanonRSMI not found; using fallback canonicalization.")

_map_re = re.compile(r":(\d+)\]")

def _mol_to_mapped_smiles(m: Chem.Mol) -> str:
    # Keep atom map numbers in SMILES
    return Chem.MolToSmiles(m, canonical=True)

def _split_mapped_rxn(mapped_rxn: str) -> Tuple[List[str], List[str]]:
    if ">>" not in mapped_rxn:
        # also accept R>G>P, we only care about R and P here
        if mapped_rxn.count(">") >= 2:
            r, _, p = mapped_rxn.split(">", 2)
            return [x for x in r.split(".") if x], [x for x in p.split(".") if x]
        raise ValueError(f"Not a reaction SMILES: {mapped_rxn}")
    r, p = mapped_rxn.split(">>")
    return [x for x in r.split(".") if x], [x for x in p.split(".") if x]

def _renumber_atom_maps(smiles: str) -> str:
    # Renumber map ids by first appearance order (stable)
    seen: Dict[str, int] = {}
    nxt = 1

    def repl(m):
        nonlocal nxt
        old = m.group(1)
        if old not in seen:
            seen[old] = nxt
            nxt += 1
        return f":{seen[old]}]"  # keep closing bracket

    return _map_re.sub(repl, smiles)

def canonicalize_mapped_rxn_fallback(mapped_rxn: str) -> str:
    rs, ps = _split_mapped_rxn(mapped_rxn)
    r_can = []
    for s in rs:
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        r_can.append(_mol_to_mapped_smiles(m))
    p_can = []
    for s in ps:
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        p_can.append(_mol_to_mapped_smiles(m))

    r_can = sorted(r_can)
    p_can = sorted(p_can)

    out = ".".join(r_can) + ">>" + ".".join(p_can)
    out = _renumber_atom_maps(out)
    return out

def canonicalize_mapped_rxn(mapped_rxn: str) -> str:
    if _HAS_SYNKIT_CANON:
        try:
            return CanonRSMI().canonicalise(mapped_rxn).canonical_rsmi  # type: ignore
        except Exception:
            # if SynKit fails for any reason, fall back
            return canonicalize_mapped_rxn_fallback(mapped_rxn)
    return canonicalize_mapped_rxn_fallback(mapped_rxn)

# Apply
dfN["rxn_mapped_canon"] = [canonicalize_mapped_rxn(x) if isinstance(x, str) else "" for x in tqdm(dfN["rxn_mapped_raw"], desc="Canonicalize")]
dfN[["rxn", "rxn_mapped_raw", "rxn_mapped_canon"]].head()


## 4. ITS construction and reaction-center extraction

We represent a mapped reaction as an **Imaginary Transition State (ITS)** graph:

- nodes: atom-map IDs
- node labels: element, formal charge, aromaticity, etc.
- edges: bond order on reactant side and product side (a *pair* label)

A bond (or atom) is in the **reaction center** if its label changes from reactants to products.

We again prefer SynKit conversion if available; otherwise we use a reference ITS builder consistent with S04.


In [ ]:
# --- Try SynKit ITS conversion first (preferred) ---
_HAS_SYNKIT_ITS = False
try:
    from synkit.IO.chem_converter import rsmi_to_its  # type: ignore
    _HAS_SYNKIT_ITS = True
    print("SynKit rsmi_to_its available.")
except Exception:
    print("SynKit rsmi_to_its not found; using fallback ITS builder.")

BondOrder = Optional[float]

def bond_order(b: Chem.Bond) -> float:
    bt = b.GetBondType()
    if bt == Chem.rdchem.BondType.SINGLE:
        return 1.0
    if bt == Chem.rdchem.BondType.DOUBLE:
        return 2.0
    if bt == Chem.rdchem.BondType.TRIPLE:
        return 3.0
    if bt == Chem.rdchem.BondType.AROMATIC:
        return 1.5
    return float(b.GetBondTypeAsDouble())

def atoms_by_map(m: Chem.Mol) -> Dict[int, Chem.Atom]:
    out: Dict[int, Chem.Atom] = {}
    for a in m.GetAtoms():
        k = int(a.GetAtomMapNum() or 0)
        if k > 0:
            out[k] = a
    return out

def bonds_by_map_pair(m: Chem.Mol) -> Dict[Tuple[int, int], float]:
    out: Dict[Tuple[int, int], float] = {}
    for b in m.GetBonds():
        a1 = b.GetBeginAtom()
        a2 = b.GetEndAtom()
        i = int(a1.GetAtomMapNum() or 0)
        j = int(a2.GetAtomMapNum() or 0)
        if i <= 0 or j <= 0:
            continue
        u, v = (i, j) if i < j else (j, i)
        out[(u, v)] = bond_order(b)
    return out

def mapped_rxn_to_its_fallback(mapped_rxn: str) -> nx.Graph:
    rs, ps = _split_mapped_rxn(mapped_rxn)
    r_mols = [Chem.MolFromSmiles(s) for s in rs]
    p_mols = [Chem.MolFromSmiles(s) for s in ps]
    r_mols = [m for m in r_mols if m is not None]
    p_mols = [m for m in p_mols if m is not None]

    # nodes: union of map ids found on either side
    atoms_r: Dict[int, Chem.Atom] = {}
    atoms_p: Dict[int, Chem.Atom] = {}
    for m in r_mols:
        atoms_r.update(atoms_by_map(m))
    for m in p_mols:
        atoms_p.update(atoms_by_map(m))

    ids = sorted(set(atoms_r.keys()) | set(atoms_p.keys()))
    G = nx.Graph()

    for k in ids:
        a = atoms_p.get(k) or atoms_r.get(k)
        # node label from whichever side exists
        G.add_node(
            k,
            symbol=a.GetSymbol() if a is not None else "?",
            formal_charge=int(a.GetFormalCharge()) if a is not None else 0,
            aromatic=bool(a.GetIsAromatic()) if a is not None else False,
        )

    bonds_r: Dict[Tuple[int, int], float] = {}
    for m in r_mols:
        bonds_r.update(bonds_by_map_pair(m))
    bonds_p: Dict[Tuple[int, int], float] = {}
    for m in p_mols:
        bonds_p.update(bonds_by_map_pair(m))

    pairs = set(bonds_r.keys()) | set(bonds_p.keys())
    for (u, v) in pairs:
        G.add_edge(u, v, r_order=bonds_r.get((u, v), None), p_order=bonds_p.get((u, v), None))
    return G

def mapped_rxn_to_its(mapped_rxn: str) -> nx.Graph:
    if _HAS_SYNKIT_ITS:
        try:
            its = rsmi_to_its(mapped_rxn)  # type: ignore
            # rsmi_to_its might return a NetworkX graph already; otherwise try attribute
            if isinstance(its, nx.Graph):
                return its
            if hasattr(its, "G"):
                return its.G
        except Exception:
            pass
    return mapped_rxn_to_its_fallback(mapped_rxn)

def reaction_center_nodes_edges(its: nx.Graph) -> Tuple[set[int], set[Tuple[int,int]]]:
    c_nodes: set[int] = set()
    c_edges: set[Tuple[int,int]] = set()
    # edge changes
    for u, v, d in its.edges(data=True):
        if d.get("r_order") != d.get("p_order"):
            c_edges.add((u, v) if u < v else (v, u))
            c_nodes.add(u); c_nodes.add(v)
    # node changes (basic): charge/aromatic/symbol changes across sides are not available in fallback;
    # If SynKit ITS encodes side-specific attrs, you can extend this.
    return c_nodes, c_edges

def extract_center_subgraph(its: nx.Graph, radius: int = 0) -> nx.Graph:
    c_nodes, c_edges = reaction_center_nodes_edges(its)
    if not c_nodes:
        return nx.Graph()
    if radius <= 0:
        nodes = set(c_nodes)
    else:
        nodes = set(c_nodes)
        frontier = set(c_nodes)
        for _ in range(radius):
            nxt = set()
            for n in frontier:
                nxt.update(its.neighbors(n))
            nxt -= nodes
            nodes |= nxt
            frontier = nxt
    return its.subgraph(nodes).copy()

# Build ITS + RC cores (radius r)
RADIUS = 0
its_list: List[nx.Graph] = []
core_list: List[nx.Graph] = []

for rxn in tqdm(dfN["rxn_mapped_canon"], desc="ITS+core"):
    if not isinstance(rxn, str) or ">" not in rxn:
        its_list.append(nx.Graph()); core_list.append(nx.Graph()); continue
    its = mapped_rxn_to_its(rxn)
    core = extract_center_subgraph(its, radius=RADIUS)
    its_list.append(its); core_list.append(core)

dfN["its"] = its_list
dfN["core"] = core_list

# Inspect core sizes
dfN["core_n_nodes"] = [g.number_of_nodes() for g in dfN["core"]]
dfN["core_n_edges"] = [g.number_of_edges() for g in dfN["core"]]
dfN[["id","class","core_n_nodes","core_n_edges"]].head()


## 5. Reaction-center clustering (WL hash prefilter + exact isomorphism)

We want to group reaction centers that are **isomorphic as typed graphs**.

To scale, we do:

1. **WL hash prefilter**: group centers by an isomorphism-invariant hash
2. **Exact isomorphism** within each bucket: NetworkX `GraphMatcher` with typed node/edge matches
3. Export: clusters with representative graphs + member indices


In [ ]:
def node_match(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    keys = ["symbol", "formal_charge", "aromatic"]
    return all(a.get(k) == b.get(k) for k in keys)

def edge_match(a: Dict[str, Any], b: Dict[str, Any]) -> bool:
    return (a.get("r_order") == b.get("r_order")) and (a.get("p_order") == b.get("p_order"))

def _decorate_for_hash(G: nx.Graph) -> nx.Graph:
    """Return a copy with single-string labels used for WL hashing."""
    H = G.copy()
    for n in H.nodes():
        sym = H.nodes[n].get("symbol")
        chg = H.nodes[n].get("formal_charge")
        aro = H.nodes[n].get("aromatic")
        H.nodes[n]["_lbl"] = f"{sym}|{chg}|{int(bool(aro))}"
    for u, v in H.edges():
        ro = H.edges[u, v].get("r_order")
        po = H.edges[u, v].get("p_order")
        H.edges[u, v]["_lbl"] = f"{ro}->{po}"
    return H

def wl_hash(G: nx.Graph) -> str:
    if G.number_of_nodes() == 0:
        return "EMPTY"
    H = _decorate_for_hash(G)
    return nx.weisfeiler_lehman_graph_hash(H, node_attr="_lbl", edge_attr="_lbl")

# 1) Bucket by WL hash
buckets: Dict[str, List[int]] = {}
for i, g in enumerate(core_list):
    h = wl_hash(g)
    buckets.setdefault(h, []).append(i)

print("Buckets:", len(buckets))
print("Largest bucket size:", max(len(v) for v in buckets.values()))

# 2) Exact clustering within each bucket (typed isomorphism)
clusters: List[List[int]] = []
reps: List[int] = []

for h, idxs in tqdm(buckets.items(), desc="Cluster buckets"):
    local_reps: List[int] = []
    local_clusters: List[List[int]] = []
    for i in idxs:
        Gi = core_list[i]
        if Gi.number_of_nodes() == 0:
            continue
        placed = False
        for rpos, rep_i in enumerate(local_reps):
            Gr = core_list[rep_i]
            GM = iso.GraphMatcher(Gr, Gi, node_match=node_match, edge_match=edge_match)
            if GM.is_isomorphic():
                local_clusters[rpos].append(i)
                placed = True
                break
        if not placed:
            local_reps.append(i)
            local_clusters.append([i])
    reps.extend(local_reps)
    clusters.extend(local_clusters)

print("Clusters:", len(clusters))
sizes = sorted([len(c) for c in clusters], reverse=True)
print("Top-10 cluster sizes:", sizes[:10])


## 6. Export a compact rule library

We export:

- `rule_library.json.gz`: one record per cluster
  - representative core graph (nodes + edges + labels)
  - member indices
  - optional metadata (class distribution, example ids)

Optionally, we also write `rules_gml/cluster_XXXX.gml` for visual inspection.


In [ ]:
def graph_to_json(G: nx.Graph) -> Dict[str, Any]:
    return {
        "nodes": [{"id": int(n), **{k: G.nodes[n].get(k) for k in ["symbol","formal_charge","aromatic"]}} for n in G.nodes()],
        "edges": [
            {
                "u": int(u),
                "v": int(v),
                "r_order": G.edges[u, v].get("r_order"),
                "p_order": G.edges[u, v].get("p_order"),
            }
            for u, v in G.edges()
        ],
    }

lib: List[Dict[str, Any]] = []

for cid, members in enumerate(clusters):
    rep = members[0]
    G = core_list[rep]

    # class histogram for the cluster (if available)
    cls = [int(dfN.iloc[i]["class"]) if "class" in dfN.columns and not pd.isna(dfN.iloc[i]["class"]) else -1 for i in members]
    hist: Dict[int, int] = {}
    for c in cls:
        hist[c] = hist.get(c, 0) + 1

    example_ids = [str(dfN.iloc[i].get("id", "")) for i in members[:5]]

    lib.append(
        {
            "cluster_id": cid,
            "size": len(members),
            "members": members,
            "rep_index": rep,
            "class_hist": hist,
            "example_ids": example_ids,
            "core": graph_to_json(G),
        }
    )

# Write JSON.GZ
lib_path = OUTDIR / "rule_library.json.gz"
_save_json_gz(lib, lib_path)
print("Wrote:", lib_path, "(clusters:", len(lib), ")")

# Optional: export representative cores as GML for browsing
max_gml = 200  # limit exports
for rec in lib[:max_gml]:
    cid = rec["cluster_id"]
    G = core_list[rec["rep_index"]]
    nx.write_gml(G, RULES_GML_DIR / f"cluster_{cid:05d}.gml")

print("Wrote GML reps (<=", max_gml, ") to:", RULES_GML_DIR)
pd.DataFrame([{"cluster_id": r["cluster_id"], "size": r["size"]} for r in lib]).sort_values("size", ascending=False).head(10)


## Discussion & extensions

**What you built:** a deduplicated library of reaction-center graphs that can be interpreted as **graph-rewriting rule cores**.

### Where SynKit fits (and what to automate next)
In production, you typically want:

- SynKit canonicalization to be the single source of truth
- SynKit ITS + center extraction (consistent labeling, more chemistry-aware)
- SynKit DPO rule export (e.g., `L|K|R` spans) rather than only center cores
- SynKit rule application engine + evaluation (connect to S06–S07)

### Exercises (recommended)
1. Increase `RADIUS` from 0 → 1 → 2 and re-run clustering. What happens to cluster counts and sizes?
2. Replace WL hash prefilter by a richer hash that includes bond-change counts.
3. Evaluate **self-reproduction**: can the representative core “explain” (match) member reactions’ cores?
